# Class 4 - Tabular Data

Today, we move from NumPy arrays to the analysis of actual tabular data (time-series from Future Forests Hartheim), before visualizing it.

Goals:
- use Pandas to analyze time-series tabular data (filter, reduce over time, summarise),
- make a first line plot with Matplotlib.


## 1. From NumPy arrays to Pandas tables

All the exercises we have seen so far use only a few data points, so one might think that using coding tools to solve them is unnecessarily complicated. 

However, imagine that instead of 20 values, you had thousands, hundreds of thousands, or even millions of data points. In such cases, these coding tools become extremely useful, and vectorization can significantly reduce the computational time required for the calculations.

Now we are finally going to move from these dummy exercises and datasets to real-world data. First, we will use the Future Forests Hartheim climate station from: 

`../processed_data/26030350_hourly.csv` # TODO: change here?

Note: we are not working directly with the raw data file from but a post-processed cleaned version with hourly data to be easier to work with.

In addition, we are going to use the library `pandas` instead of `NumPy` because `pandas` provides convenient and more powerfull tools for working with structured, time-series datasets, such as selecting columns, filtering data, handling missing values, and working with timestamps.


In [ ]:
import pandas as pd


### i. Loading a CSV with `pd.read_csv()`

`pd.read_csv()` reads a CSV file and returns a `DataFrame`, the central object in Pandas. A `DataFrame` is a table where each column can hold a different type (numbers, strings, dates…) and every row shares the same index. You can think of it as a spreadsheet inside Python.

By default, date/time columns are read as plain text. The argument `parse_dates=['utc_time']` tells Pandas to convert that column into proper datetime objects, which lets us later filter by date ranges and resample in time.


In [ ]:
HartheimRaw = pd.read_csv('../processed_data/26030350_hourly.csv', parse_dates=['utc_time'])
HartheimRaw = HartheimRaw.set_index('utc_time') # turns the utc_time column into the row labels (index)
print(type(HartheimRaw))   # Print the type of the object created by read_csv() -> pandas dataframe

# This is always what I print when I make a DataFrame
print(HartheimRaw.shape)   # Print shape (size) of the dataframe
print(HartheimRaw.columns) # Print the headers (columns)

In [ ]:
# Printing the full DataFrame is too big
print(HartheimRaw)

As you can see we moved from data with less than 10 data points to a dataset with 330*22 = 7.260 entries.

### ii. Basic Pandas operations

#### a. Column selection

A DataFrame often has more columns than you need. You can select a subset by passing a list of column names.


In [ ]:
VariablesToKeep = [         # A python list of strings, I just copy the columns I need from the previous cell
    'temperature_2m',
    'humidity_2m',
    'temperature_ground',
    'precipitation'
]
HartheimData = HartheimRaw[VariablesToKeep] # Filter the DataFrame with the variables to keep
print(HartheimData.columns)

#### b. Indexing and slicing

In `NumPy`, indexing is almost always by **position**: `Array[0]` is the first element, and `Array[1:5]` selects positions 1 to 4 (stop excluded).

In `pandas`, there are two ways to do indexing and slicing,

- by **position** (row/column numbers) with `.iloc[row_index, column_index]` (similar to `Numpy`):


In [ ]:
print(HartheimData.iloc[0, 1])        # first row, second column
print(HartheimData.iloc[0:3, 1:3])    # rows 0-2, columns 1-2 (stop excluded, like NumPy)


- or by **label** (names / dates) with `.loc[row_names, column_names]`. But before being able to use label, we need labels for our rows. `.set_index('utc_time')` moves the time column from a regular column to the row index. Once the index is a datetime, you can slice by date strings with `.loc['start':'end']` (both ends included). This is much more readable than writing manual comparisons.


In [ ]:
print(HartheimData.loc['2026-07-24':'2026-07-26', ['temperature_2m', 'temperature_ground']])

⚠️ Important difference: with `.loc`, a slice like `'2026-07-24':'2026-07-26'` is **inclusive on both ends**. In `NumPy`, the stop index is excluded.

#### c. Creating a new column:

You can create a new column by assigning the result of an operation between existing columns. Here we compute the vertical temperature gradient (2 m minus ground). The operation is vectorised, just like in NumPy.


In [ ]:
HartheimData['temp_gradient_2m_minus_ground'] = HartheimData['temperature_2m'] - HartheimData['temperature_ground']

HartheimData[['temperature_2m', 'temperature_ground', 'temp_gradient_2m_minus_ground']].head()

#### d. Summary statistics

`.describe()` gives you count, mean, standard deviation, min, max, and quartiles in one call. Pandas skips `NaN` values automatically, so you get correct statistics even when some measurements are missing.


In [ ]:
print(HartheimData[['temperature_2m', 'humidity_2m', 'precipitation']].describe())

### iii. Vectorisation operations with Pandas

In NumPy we used vectorised formulas, we can do exactly the same on Pandas columns.


In [ ]:
# Celsius -> Fahrenheit (same as NumPy)
HartheimData['temperature_2m_f'] = HartheimData['temperature_2m'] * 9 / 5 + 32

In [ ]:
# Mean is used at the end and not with np.mean()
MeanC = HartheimData['temperature_2m'].mean()
MeanF = HartheimData['temperature_2m_f'].mean()

print(f'Mean 2m temperature: {MeanC:.2f} °C')
print(f'Mean 2m temperature: {MeanF:.2f} °F')

### iv. Filtering rows

Filtering keeps only the rows that match a condition. In Pandas this uses a **Boolean mask**: a column of `True`/`False` values, one per row. You write the condition, then put the mask inside `[]`.

In [ ]:
# One condition: temperatures warmer than 38 °C
HotTemp = HartheimData[HartheimData['temperature_2m'] > 38]
print('Hot temperatures (>38°C):', HotTemp['temperature_2m'])

Try to print the **number of hours** with cold temperatures (colder than 10°C)

In [ ]:
# Solution
print(len(HartheimData[HartheimData['temperature_2m'] < 10]))

In [ ]:
# Several conditions: warm AND humid (parentheses are required with & / |)
WarmHumid = HartheimData[
    (HartheimData['temperature_2m'] > 25) & (HartheimData['humidity_2m'] > 60)
]
print('Warm and humid hours:', len(WarmHumid))

In [ ]:
# Same idea with a Boolean mask on the index
LateJulyMask = HartheimData[
    (HartheimData.index >= '2026-07-24') & (HartheimData.index <= '2026-07-26')
]
print('Rows with index mask:', len(LateJulyMask))

#### a. Getting row labels with `.index`

Every DataFrame has an **index**: the label for each row. After `set_index('utc_time')`, the index holds the UTC timestamps.

- `HartheimData.index` — all row labels (here: UTC times)
- `df.index[mask]` — keep only the labels where the Boolean mask is `True`

This is useful when you want the **times** that match a condition, not the full rows.

In [ ]:
WarmHumidMask = (HartheimData['temperature_2m'] > 25) & (HartheimData['humidity_2m'] > 60)
print('UTC times warm and humid:')
print(HartheimData.index[WarmHumidMask])

### v. Changing time resolution (`resample`)

Sometimes we want to change the time resolution of a series. For example, roll hourly measurements up to daily values. **Resampling** groups rows into coarser time bins (e.g. days) and then applies an aggregation such as mean or sum.

Because the index is already a datetime, you can write:

- `.resample('D').mean()` — daily average (good for temperature, humidity)
- `.resample('D').sum()` — daily total (good for precipitation)

`'D'` means calendar day. Other common frequencies: `'h'` (hour), `'W'` (week), `'ME'` (month end) or `'YE'` (year end).

In [ ]:
# Daily mean of continuous variables
DailyMean = HartheimData[['temperature_2m', 'humidity_2m']].resample('D').mean()
print('Before:')
print(HartheimData)
print('After:')
print(DailyMean)

In [ ]:
# Daily sum of precipitation (mm per day)
DailyPrecip = HartheimData[['precipitation']].resample('D').sum()
print('\nDaily precipitation totals (mm):')
print(DailyPrecip)

Try to print the weekly mean temperature and total precipitation

In [ ]:
# solution
subset = HartheimData.loc['2026-07-20':'2026-08-02']
print(subset[['temperature_2m']].resample('W').mean())
print(subset[['precipitation']].resample('W').sum())

## Exercise A

Root growth and soil microbial activity often depend on how warm the upper soil layers are compared with deeper layers. You are investigating whether the upper soil layer becomes substantially warmer than the deeper soil layer during warm conditions at the Hartheim station.

The logger measures soil temperature at two depths: about 0.2 m (`soil_temperature_0_2`) and about 0.6 m (`soil_temperature_0_6`).

Tasks:
1. Compute the mean and standard deviation of both soil temperature columns (`soil_temperature_0_2`, `soil_temperature_0_6`). Look up online for how to use standard deviation in `Pandas`
2. For how many hours is the 0.2 m temperature higher than the 0.6 m temperature, and for how many hours is the opposite true? 
3. Print the UTC times of all hours corresponding to the condition that occurs for the fewer number of hours.

In [ ]:
# Solution

# 1) Mean and standard deviation of both soil temperature columns
SoilTemps = HartheimRaw[['soil_temperature_0_2', 'soil_temperature_0_6']]
print('Mean shallow (0.2 m):', SoilTemps['soil_temperature_0_2'].mean())
print('Std  shallow (0.2 m):', SoilTemps['soil_temperature_0_2'].std())
print('Mean deep (0.6 m):', SoilTemps['soil_temperature_0_6'].mean())
print('Std  deep (0.6 m):', SoilTemps['soil_temperature_0_6'].std())

# 2) Compare depths hour by hour
ShallowWarmer = SoilTemps['soil_temperature_0_2'] > SoilTemps['soil_temperature_0_6']
DeepWarmer = SoilTemps['soil_temperature_0_6'] > SoilTemps['soil_temperature_0_2']

print('Hours with shallow (0.2 m) warmer than deep (0.6 m):', ShallowWarmer.sum())
print('Hours with deep (0.6 m) warmer than shallow (0.2 m):', DeepWarmer.sum())

# 3) UTC times when deep soil is warmer than shallow soil
print(SoilTemps.index[DeepWarmer])

## 2. Plotting: Matplotlib essentials

Matplotlib is the most widely used plotting library in Python. We only need a handful of functions to make a clean line graph. 


In [ ]:
import matplotlib.pyplot as plt

Start with a plot type function:

- `plt.plot(x, y, label='...')`, draws a line from the x and y data. The `label` is used by the legend. 

Look on the online documentation to see all the different plot types you can make

Then you have different **optional** options to choose:
- `plt.title('...')` — sets the plot title.
- `plt.xlabel('...')` / `plt.ylabel('...')` — set the axis labels.
- `plt.grid(True)` — adds a background grid for easier reading.
- `plt.legend()` — displays a legend using the labels defined in each `plt.plot()` call.

Finally, you can display the figure with `plt.show()`.


In [ ]:
# Select the data to plot
PlotDf = HartheimData.loc['2026-07-24':'2026-07-27']

# Draw one line per variable; the index (datetime) is the x-axis
plt.plot(PlotDf.index, PlotDf['temperature_2m'], label='Temperature 2m')
plt.plot(PlotDf.index, PlotDf['temperature_ground'], label='Temperature ground')

# The next lines are optional, they "dress up" the graph
plt.title('Hourly temperatures (2026-07-24 to 2026-07-27)')  # Title above the plot
plt.xlabel('UTC time')          # Label for the x-axis
plt.ylabel('Temperature (°C)')  # Label for the y-axis
plt.grid(True)                  # Add a background grid
plt.legend()                    # Show the legend built from the labels above

plt.show()                      # Render the figure

## Exercise B

After a dry period, a colleague asks whether rainy hours at Hartheim also come with higher near-surface humidity. You decide to check this visually over the full measurement period, because a simple time series plot is often the fastest way to spot co-varying weather patterns.

Using the Hartheim dataset, plot precipitation and 2 m humidity (tip: one per code cell), can you see a relationship between both?


In [ ]:
# Solution

# Precipitation
plt.plot(HartheimData.index, HartheimData['precipitation'], color='blue')
plt.xlabel('UTC time')
plt.ylabel('Precipitation (mm)')

plt.show()

In [ ]:
# Solution

# Humidity
plt.plot(HartheimData.index, HartheimData['humidity_2m'], color='orange')
plt.title('Humidity 2m')
plt.xlabel('UTC time')
plt.ylabel('Humidity 2m (%)')

plt.show()

## Exercise C (Bonus)

Trees do not close their stomata because the air is hot, but because the air is *dry*. The relevant variable is the vapour pressure deficit (VPD): the difference between how much water vapour the air could hold and how much it actually holds.

You can compute VPD from the station data with the Tetens formula:

$$e_s = 0.6108 \cdot \exp\left(\frac{17.27\,T}{T + 237.3}\right) \qquad \text{VPD} = e_s \cdot \left(1 - \frac{RH}{100}\right)$$

with $T$ the air temperature in °C, $RH$ the relative humidity in %, and $e_s$ and VPD in kPa.

**Question: is the hottest day at Hartheim also the most physiologically stressful one for the trees?**

Tasks:

1. Using `temperature_2m` and `humidity_2m`, create a new column `vpd_kpa`. Do it in two vectorised steps (first `es_kpa`, then `vpd_kpa`). You will need `np.exp()`, which works directly on a Pandas column.

2. Print the mean, minimum and maximum of `vpd_kpa`, and the UTC time at which the maximum occurs. Tip: you already know how to build a mask with `>`. A mask with `==` works the same way, so compare the column to its own maximum and pass the mask to `.index[...]`.

3. A VPD above 3 kPa is often used as an indicative threshold above which stomatal conductance in temperate conifers is strongly reduced. How many hours of the record are above it? Plot VPD, and add a horizontal line at 3 kPa (look up `plt.axhline` in the Matplotlib documentation). In another cell, plot the temperature.

4. Compute two daily series with `resample`: the daily mean of `temperature_2m` and the daily maximum of `vpd_kpa`. Compare the two rankings: is the day with the highest mean temperature also the day with the highest maximum VPD?

In [ ]:
VpdData = HartheimRaw.copy()

# Solution

# 1) VPD from temperature and humidity (two vectorised steps)
VpdData['es_kpa'] = 0.6108 * np.exp(
    17.27 * VpdData['temperature_2m'] / (VpdData['temperature_2m'] + 237.3)
)
VpdData['vpd_kpa'] = VpdData['es_kpa'] * (1 - VpdData['humidity_2m'] / 100)

In [ ]:
# 2) Basic statistics and timing of the maximum
print("Mean VPD: (in kPa)", VpdData['vpd_kpa'].mean())
print("Min  VPD: (in kPa)", VpdData['vpd_kpa'].min())
print("Max  VPD: (in kPa)", VpdData['vpd_kpa'].max())

print('Time of maximum VPD:', VpdData.index[VpdData['vpd_kpa'] == VpdData['vpd_kpa'].max()])

In [ ]:
# 3) Hours above the stress threshold
StressMask = VpdData['vpd_kpa'] > 3
print('Number of stressful hours (VPD > 3 kPa):', len(VpdData[StressMask])) # Also StressMask.sum()

plt.plot(VpdData['vpd_kpa'], label='VPD')
plt.axhline(3, color='red', linestyle='--', label='Stress threshold (3 kPa)')
plt.xlabel('UTC time')
plt.ylabel('VPD (kPa)')
plt.legend()
plt.show()

In [ ]:
plt.plot(VpdData['temperature_2m'], label='Temperature at 2m')
plt.xlabel('UTC time')
plt.ylabel('Temperature (in °C)')
plt.legend()
plt.show()

In [ ]:
# 4) Daily aggregation on two different reducers
DailyMeanTemp = VpdData[['temperature_2m']].resample('D').mean()
DailyMaxVpd = VpdData[['vpd_kpa']].resample('D').max()
MaxTemp = DailyMeanTemp['temperature_2m'].max()
MaxDailyVpd = DailyMaxVpd['vpd_kpa'].max()
print('Warmest day (mean T):', DailyMeanTemp.index[DailyMeanTemp['temperature_2m'] == MaxTemp])
print('Most stressful day (max VPD):', DailyMaxVpd.index[DailyMaxVpd['vpd_kpa'] == MaxDailyVpd])